# 실습 11: 조정 다이얼 찾기
- 상황: 모델에는 사람이 정해줘야 하는 값이 있는데, 지금까지 손대지 않고 썼다
- 목표: 그 값을 손으로 돌려보고, 자동 탐색으로 찾아본다

## Step 0. 앞 실습까지 재현하기

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

# 1. 불러오기
df = pd.read_csv("../../day02/lab06_clean-dataset/results/secom_clean.csv")

# 2. 센서 열 빈칸을 중앙값으로 채우기
sensor_cols = [c for c in df.columns if c != "result"]
df[sensor_cols] = df[sensor_cols].fillna(df[sensor_cols].median())

# 3. 불량여부 열 만들기
df["불량여부"] = (df["result"] == "불량").astype(int)

# 4. X, y 나누기
X = df[sensor_cols]
y = df["불량여부"]

# 5. 학습용·시험용 나누기
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 6. 의사결정나무 - class_weight="balanced"만 추가, 다른 설정은 기본값
tree = DecisionTreeClassifier(class_weight="balanced", random_state=42)
tree.fit(X_train, y_train)
예측 = tree.predict(X_test)

맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 예측).ravel()

print("정확도:", round(accuracy_score(y_test, 예측) * 100, 2), "%")
print("잡은 불량:", 잡은불량, "/ 놓친 불량:", 놓친불량, "/ 헛경보:", 헛경보)

정확도: 88.85 %
잡은 불량: 2 / 놓친 불량: 19 / 헛경보: 16


## Step 4. 설정값 자동 탐색하기_재현률

In [2]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import recall_score, precision_score, f1_score

# 후보 설정값 - max_depth, min_samples_leaf만 바꿔가며 찾는다
설정후보 = {
    "max_depth": [2, 3, 4, 5, 10, None],
    "min_samples_leaf": [1, 5, 10, 20],
}

# 학습용만 써서 탐색한다 (cv로 학습용을 다시 나눠 채점하므로 시험용은 쓰지 않는다)
탐색 = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    param_grid=설정후보,
    scoring="recall",
    cv=5,
)
탐색.fit(X_train, y_train)

print("1등 설정값:", 탐색.best_params_)
print("탐색 중 나온 점수(재현율):", round(탐색.best_score_, 3))

# 1등 설정으로 이미 학습된 모델을 그대로 가져와 시험용을 채점한다
최고모델 = 탐색.best_estimator_
예측 = 최고모델.predict(X_test)

맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 예측).ravel()

print()
print("정확도:", round(accuracy_score(y_test, 예측) * 100, 2), "%")
print("잡은 불량:", 잡은불량, "/ 놓친 불량:", 놓친불량, "/ 헛경보:", 헛경보)
print("재현율:", round(recall_score(y_test, 예측), 3))
print("정밀도:", round(precision_score(y_test, 예측, zero_division=0), 3))
print("F1:", round(f1_score(y_test, 예측), 3))

1등 설정값: {'max_depth': 3, 'min_samples_leaf': 20}
탐색 중 나온 점수(재현율): 0.53

정확도: 48.09 %
잡은 불량: 14 / 놓친 불량: 7 / 헛경보: 156
재현율: 0.667
정밀도: 0.082
F1: 0.147


이번에는 1등을 뽑는 기준만 F1 으로 

In [3]:
# 방금과 똑같은 후보로, 1등을 뽑는 기준만 F1으로 바꿔서 다시 탐색한다
탐색_f1 = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    param_grid=설정후보,
    scoring="f1",
    cv=5,
)
탐색_f1.fit(X_train, y_train)

print("1등 설정값(F1 기준):", 탐색_f1.best_params_)
print("탐색 중 나온 점수(F1):", round(탐색_f1.best_score_, 3))

최고모델_f1 = 탐색_f1.best_estimator_
예측_f1 = 최고모델_f1.predict(X_test)

맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 예측_f1).ravel()

print()
print("정확도:", round(accuracy_score(y_test, 예측_f1) * 100, 2), "%")
print("잡은 불량:", 잡은불량, "/ 놓친 불량:", 놓친불량, "/ 헛경보:", 헛경보)
print("재현율:", round(recall_score(y_test, 예측_f1), 3))
print("정밀도:", round(precision_score(y_test, 예측_f1, zero_division=0), 3))
print("F1:", round(f1_score(y_test, 예측_f1), 3))

print()
print(f"{'기준':10} {'설정값':40} {'정확도':>8} {'재현율':>8} {'정밀도':>8} {'F1':>8}")
print(f"{'재현율':10} {str(탐색.best_params_):40} {round(accuracy_score(y_test, 예측)*100,2):>7}% {recall_score(y_test, 예측):>8.3f} {precision_score(y_test, 예측, zero_division=0):>8.3f} {f1_score(y_test, 예측):>8.3f}")
print(f"{'F1':10} {str(탐색_f1.best_params_):40} {round(accuracy_score(y_test, 예측_f1)*100,2):>7}% {recall_score(y_test, 예측_f1):>8.3f} {precision_score(y_test, 예측_f1, zero_division=0):>8.3f} {f1_score(y_test, 예측_f1):>8.3f}")

1등 설정값(F1 기준): {'max_depth': 10, 'min_samples_leaf': 10}
탐색 중 나온 점수(F1): 0.192

정확도: 76.75 %
잡은 불량: 6 / 놓친 불량: 15 / 헛경보: 58
재현율: 0.286
정밀도: 0.094
F1: 0.141

기준         설정값                                           정확도      재현율      정밀도       F1
재현율        {'max_depth': 3, 'min_samples_leaf': 20}   48.09%    0.667    0.082    0.147
F1         {'max_depth': 10, 'min_samples_leaf': 10}   76.75%    0.286    0.094    0.141


## Step 5. 로지스틱 회귀의 사람이 정해주는 값 - C 바꿔보기

로지스틱 회귀에도 사람이 미리 정해주는 값(하이퍼파라미터)이 있다: `C`, `max_iter`, `class_weight`, `penalty`, `solver` 등.
그중 `C`를 0.01 / 0.1 / 1 / 10 으로 바꿔가며 학습용으로 학습, 시험용으로 채점한다.

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
X_train_스케일 = scaler.fit_transform(X_train)
X_test_스케일 = scaler.transform(X_test)

결과 = []
for C값 in [0.01, 0.1, 1, 10]:
    로지스틱모델 = LogisticRegression(max_iter=1000, class_weight="balanced", C=C값)
    로지스틱모델.fit(X_train_스케일, y_train)
    로지스틱예측 = 로지스틱모델.predict(X_test_스케일)

    맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 로지스틱예측).ravel()
    정확도 = accuracy_score(y_test, 로지스틱예측)
    재현율 = recall_score(y_test, 로지스틱예측, zero_division=0)
    정밀도 = precision_score(y_test, 로지스틱예측, zero_division=0)
    f1 = f1_score(y_test, 로지스틱예측, zero_division=0)
    결과.append((C값, 정확도, 잡은불량, 헛경보, 재현율, 정밀도, f1))

print(f"{'C':>6} {'정확도':>8} {'잡은불량':>8} {'헛경보':>8} {'재현율':>8} {'정밀도':>8} {'F1':>8}")
for C값, 정확도, 잡은불량, 헛경보, 재현율, 정밀도, f1 in 결과:
    print(f"{C값:>6} {round(정확도*100,2):>7}% {잡은불량:>8} {헛경보:>8} {재현율:>8.3f} {정밀도:>8.3f} {f1:>8.3f}")

     C      정확도     잡은불량      헛경보      재현율      정밀도       F1
  0.01   75.16%       13       70    0.619    0.157    0.250
   0.1   73.57%       10       72    0.476    0.122    0.194
     1   74.84%       10       68    0.476    0.128    0.202
    10   73.89%       10       71    0.476    0.123    0.196


### 결과 정리 (실행 결과 기준)

C는 모델이 학습 데이터에 얼마나 딱 맞추려 하는지(작을수록 단순하게, 클수록 세밀하게 맞추려는 정도)를 조절하는 값이다.

| C | 정확도 | 잡은불량 | 헛경보 | 재현율 | 정밀도 | F1 |
|---|---|---|---|---|---|---|
| 0.01 | 75.16% | 13 | 70 | 0.619 | 0.157 | 0.250 |
| 0.1 | 73.57% | 10 | 72 | 0.476 | 0.122 | 0.194 |
| 1 | 74.84% | 10 | 68 | 0.476 | 0.128 | 0.202 |
| 10 | 73.89% | 10 | 71 | 0.476 | 0.123 | 0.196 |